In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score


data = pd.read_excel(r"C:\Users\jitis\Downloads\breast_cancer_dataset (1).xlsx")


X = data.drop(["target", "diagnosis"], axis=1)
y = data["target"]

print("Dataset loaded successfully")
print("Total number of rows:", len(data))
print("Total number of features:", X.shape[1])


X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42,
    stratify=y
)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print("\nData split completed")
print("Training data:", X_train.shape[0])
print("Validation data:", X_val.shape[0])
print("Testing data:", X_test.shape[0])



baseline_model = LogisticRegression(max_iter=5000)


baseline_model.fit(X_train, y_train)


baseline_predictions = baseline_model.predict(X_val)


baseline_accuracy = accuracy_score(y_val, baseline_predictions)

print("\nBaseline Logistic Regression")
print("Validation Accuracy:", round(baseline_accuracy, 4))



cv_scores = cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("\n5-Fold Cross-Validation Results")
print("Scores:", cv_scores)
print("Average Accuracy:", round(cv_scores.mean(), 4))
print("Standard Deviation:", round(cv_scores.std(), 4))



pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=5000))
])


pipeline.fit(X_train, y_train)


pipeline_predictions = pipeline.predict(X_val)


pipeline_accuracy = accuracy_score(y_val, pipeline_predictions)

print("\nScaled Logistic Regression Pipeline")
print("Validation Accuracy:", round(pipeline_accuracy, 4))



pipeline_cv_scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("\nPipeline 5-Fold CV Scores:", pipeline_cv_scores)
print("Average Pipeline CV Accuracy:",
      round(pipeline_cv_scores.mean(), 4))



parameter_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100]
}


grid_search = GridSearchCV(
    pipeline,
    parameter_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)


grid_search.fit(X_train, y_train)


best_model = grid_search.best_estimator_

print("\nGridSearchCV Results")
print("Best C value:", grid_search.best_params_)
print("Best Cross-Validation Accuracy:",
      round(grid_search.best_score_, 4))



tuned_predictions = best_model.predict(X_val)


tuned_accuracy = accuracy_score(y_val, tuned_predictions)

print("\nModel Comparison")
print("Baseline Accuracy:", round(baseline_accuracy, 4))
print("Pipeline Accuracy:", round(pipeline_accuracy, 4))
print("Tuned Model Accuracy:", round(tuned_accuracy, 4))



test_predictions = best_model.predict(X_test)


test_accuracy = accuracy_score(y_test, test_predictions)

print("\nFinal Test Result")
print("Test Accuracy:", round(test_accuracy, 4))

Dataset loaded successfully
Total number of rows: 569
Total number of features: 30

Data split completed
Training data: 341
Validation data: 114
Testing data: 114

Baseline Logistic Regression
Validation Accuracy: 0.9649

5-Fold Cross-Validation Results
Scores: [0.92753623 0.94117647 0.92647059 0.98529412 0.91176471]
Average Accuracy: 0.9384
Standard Deviation: 0.0252

Scaled Logistic Regression Pipeline
Validation Accuracy: 0.9912

Pipeline 5-Fold CV Scores: [0.95652174 0.97058824 0.95588235 0.97058824 0.98529412]
Average Pipeline CV Accuracy: 0.9678

GridSearchCV Results
Best C value: {'classifier__C': 0.1}
Best Cross-Validation Accuracy: 0.9736

Model Comparison
Baseline Accuracy: 0.9649
Pipeline Accuracy: 0.9912
Tuned Model Accuracy: 0.9825

Final Test Result
Test Accuracy: 0.9825
